# Document Q&A (mini RAG) — Kaggle, no API key

Ask questions about one or more PDF files using entirely local, open-source models. The notebook creates embeddings with `all-MiniLM-L6-v2`, searches them with FAISS, and produces an answer with `Qwen/Qwen2.5-1.5B-Instruct`.

**No OpenAI, Hugging Face, or other API key is required.** Enable Internet only for the first run so Kaggle can download packages and public model weights. After saving the models as a Kaggle Dataset, you can run with Internet disabled.

## Kaggle setup

1. Create a new Kaggle Notebook and turn on a **GPU** accelerator (T4 is enough).
2. Create a Kaggle Dataset containing your PDF files, then attach it through **Add Input**.
3. Turn on Internet for this first run.
4. Run every cell in order. In the next cell, replace `YOUR_DATASET_FOLDER` with the folder name shown under `/kaggle/input/`.

Do not upload confidential documents to a public Kaggle dataset.

In [5]:
# Install the small libraries not guaranteed to be preinstalled in every Kaggle image.
!pip -q install -U sentence-transformers faiss-cpu pypdf accelerate

In [6]:
from pathlib import Path
import re
import numpy as np
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# Change this to your attached Kaggle Dataset folder.
DOCUMENT_FOLDER = Path('/kaggle/input/datasets/vaidiknakarani/project-list-3rd-year')
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
TOP_K = 4

assert torch.cuda.is_available(), 'Enable GPU in Kaggle: Settings → Accelerator → GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## Read and chunk the PDFs

A chunk is a small overlapping piece of text. The overlap prevents important context from being split between chunks.

In [7]:
def extract_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            pages.append({'source': pdf_path.name, 'page': page_number, 'text': text})
    return pages

def chunk_text(text, chunk_size=900, overlap=160):
    words = text.split()
    step = chunk_size - overlap
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), step) if words[i:i + chunk_size]]

pdf_files = sorted(DOCUMENT_FOLDER.rglob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDFs found in {DOCUMENT_FOLDER}. Check DOCUMENT_FOLDER and attach your dataset.')

chunks = []
for pdf_file in pdf_files:
    for page_data in extract_pdf_pages(pdf_file):
        for text_chunk in chunk_text(page_data['text']):
            chunks.append({**page_data, 'text': text_chunk})

print(f'Loaded {len(pdf_files)} PDF(s) and created {len(chunks)} chunks.')
print('Example:', chunks[0]['text'][:300])

Loaded 1 PDF(s) and created 42 chunks.
Example: AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch AI/ML Capstone Project Problem Statements Parul University in collaboration with TelcoLearn B.Tech 3rd/4th Year — 2027 Graduating Batch — Phase 2 Bootcamp Arpit Tripathi & Sanjay Kumar Instructions for Students Team


## Create the searchable vector index

The embedding model converts each chunk into numbers. FAISS finds chunks that are closest in meaning to a question.

In [8]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device='cuda')
embeddings = embedder.encode(
    [item['text'] for item in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product = cosine similarity after normalization
index.add(embeddings)
print(f'FAISS index contains {index.ntotal} chunks.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index contains 42 chunks.


## Load the local language model

This downloads a public 1.5B-parameter model on the first run. It runs locally in the Kaggle session; your question is not sent to an API.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
).eval()
print('Model loaded.')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [10]:
def retrieve(question, top_k=TOP_K):
    query = embedder.encode([question], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(query, top_k)
    return [(chunks[i], float(score)) for i, score in zip(ids[0], scores[0])]

def answer_question(question, top_k=TOP_K, max_new_tokens=350):
    results = retrieve(question, top_k)
    context = '\n\n'.join(
        f'[{n}] Source: {item["source"]}, page {item["page"]}\n{item["text"]}'
        for n, (item, _) in enumerate(results, start=1)
    )
    messages = [
        {'role': 'system', 'content': 'Answer only from the supplied context. If the answer is not in the context, say: I could not find that in the documents. Cite sources like [1] or [2].'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer_tokens = generated[0][model_inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    print('Answer:\n', answer)
    print('\nRetrieved passages:')
    for n, (item, score) in enumerate(results, start=1):
        print(f'[{n}] {item["source"]}, page {item["page"]} (similarity {score:.3f})')
    return answer, results

# Replace with a question about your documents.
answer_question('What are the most important points in these documents?')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer:
 The most important points in these documents are:

1. **Legal Document Summarization and Q&A System**:
   - **Domain**: NLP
   - **Dataset**: Kaggle: anaderm/indian-court-cases
   - **Problem Statement**: Build a web application for law students and paralegals that accepts a legal document upload, automatically classifies the case type, summarizes the document, discovers key legal themes, and answers natural-language questions about the document using retrieval-augmented generation.

2. **Fake News Detection and Fact-Checking Assistant**:
   - **Domain**: NLP
   - **Dataset**: ISOT Fake and Real News Dataset, Kaggle (clmentbisaillon/fake-and-real-news-dataset)
   - **Problem Statement**: Build a browser-accessible web application that allows users to paste a news article or headline and receive an authenticity assessment with a confidence score, topic analysis showing whether the article falls into a known disinformation topic cluster, and an LLM-powered fact-check with source

('The most important points in these documents are:\n\n1. **Legal Document Summarization and Q&A System**:\n   - **Domain**: NLP\n   - **Dataset**: Kaggle: anaderm/indian-court-cases\n   - **Problem Statement**: Build a web application for law students and paralegals that accepts a legal document upload, automatically classifies the case type, summarizes the document, discovers key legal themes, and answers natural-language questions about the document using retrieval-augmented generation.\n\n2. **Fake News Detection and Fact-Checking Assistant**:\n   - **Domain**: NLP\n   - **Dataset**: ISOT Fake and Real News Dataset, Kaggle (clmentbisaillon/fake-and-real-news-dataset)\n   - **Problem Statement**: Build a browser-accessible web application that allows users to paste a news article or headline and receive an authenticity assessment with a confidence score, topic analysis showing whether the article falls into a known disinformation topic cluster, and an LLM-powered fact-check with sou

## Optional: ask more questions

Run the next cell repeatedly with a new question. The model and index stay in memory, so later answers are faster.

### Troubleshooting
- **Out of memory:** change `LLM_MODEL` to `Qwen/Qwen2.5-0.5B-Instruct` or reduce `TOP_K` to 3.
- **No PDFs found:** correct `DOCUMENT_FOLDER`; the exact folder name appears under `/kaggle/input`.
- **Scanned PDF gives blank text:** OCR it before using this notebook, because `pypdf` reads embedded text rather than images.
- **Internet-off run:** save the model folders as a private Kaggle Dataset and set model paths to `/kaggle/input/<your-model-dataset>/<folder>`.

In [11]:
question = 'What dataset is used for the Chest X-Ray Pneumonia Detection project?'
answer_question(question)

Answer:
 The dataset used for the Chest X-Ray Pneumonia Detection project is available on Kaggle under the name "paultimothymooney/chest-xray-pneumonia".

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 3 (similarity 0.429)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 40 (similarity 0.388)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 2 (similarity 0.386)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 14 (similarity 0.352)


('The dataset used for the Chest X-Ray Pneumonia Detection project is available on Kaggle under the name "paultimothymooney/chest-xray-pneumonia".',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 3,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem Technique App component A Extract HOG and pixel histogram features; train SVM and Random Forest baseline. Establish non-deep-learning accuracy ceiling. Supervised, feature engineering Backend baseline comparison tab B Train CNN Autoencoder on normal X-rays only; flag high reconstruction error as anomalous (no disease labels used). Unsupervised anomaly detection Anomaly score overlay C Fine-tune DenseNet-121 on the full labelled dataset. Implement Grad-CAM heatmap visualisation on predicted images. Transfer learning, XAI Image upload + heatmap render D Build a radiology report generator: given prediction confidence and detected regions, an LLM produces a s

In [12]:
question = 'What are the four deliverables required for each capstone project?'
answer_question(question)

Answer:
 The four deliverables required for each capstone project are:

1. **Kaggle/Colab notebooks** for each sub-problem (reproducible, with outputs saved)
2. **A deployed web or mobile application with a public URL or APK**
3. **A 10-minute demo video walking through the application and model results**
4. **A 4-page technical report (IEEE format) with model comparisons and architecture diagram**

These deliverables cover both the theoretical aspects (technical reports) and practical implementation details (deployed applications) for each sub-problem within the capstone projects.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 1 (similarity 0.453)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.365)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 36 (similarity 0.272)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 11 (similarity 0.262)


('The four deliverables required for each capstone project are:\n\n1. **Kaggle/Colab notebooks** for each sub-problem (reproducible, with outputs saved)\n2. **A deployed web or mobile application with a public URL or APK**\n3. **A 10-minute demo video walking through the application and model results**\n4. **A 4-page technical report (IEEE format) with model comparisons and architecture diagram**\n\nThese deliverables cover both the theoretical aspects (technical reports) and practical implementation details (deployed applications) for each sub-problem within the capstone projects.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 1,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch AI/ML Capstone Project Problem Statements Parul University in collaboration with TelcoLearn B.Tech 3rd/4th Year — 2027 Graduating Batch — Phase 2 Bootcamp Arpit Tripathi & Sanjay Kumar Instructions for Students Team composition: Each project

In [13]:
question = 'Which project uses the CICIDS2017 dataset?'
answer_question(question)

Answer:
 The project that uses the CICIDS2017 dataset is "Network Intrusion Detection System" problem statement under the domain of Cybersecurity.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.372)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 36 (similarity 0.370)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 38 (similarity 0.362)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 41 (similarity 0.356)


('The project that uses the CICIDS2017 dataset is "Network Intrusion Detection System" problem statement under the domain of Cybersecurity.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 42,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch LLM API: Google AI Studio (Gemini 1.5 Flash) provides a free tier sufficient for all Student D sub-problems. Deployment: Render.com, Railway.app, and HuggingFace Spaces all offer free hosting tiers for student projects. Page 42 of 42'},
   0.3724830150604248),
  ({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 36,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem Technique App component A Handle 79 mixed features (36 numeric, 43 categorical). Train Ridge, Lasso, XGBoost. Log-transform price. Target: competitive RMSLE on Kaggle leaderboard. Supervised regression Price prediction API B Cluster proper

In [14]:
question = 'Which projects use K-Means clustering for Students sub-problem?'
answer_question(question)

Answer:
 Based on the provided information:

- **Project D** uses K-Means clustering for the "Cluster students by learning behavior" sub-problem.

This can be seen from the description of Project D's problem statement which mentions clustering students by their learning behavior using K-Means on submission timestamps and engagement data.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.470)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 38 (similarity 0.449)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 21 (similarity 0.402)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 1 (similarity 0.399)


('Based on the provided information:\n\n- **Project D** uses K-Means clustering for the "Cluster students by learning behavior" sub-problem.\n\nThis can be seen from the description of Project D\'s problem statement which mentions clustering students by their learning behavior using K-Means on submission timestamps and engagement data.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 42,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch LLM API: Google AI Studio (Gemini 1.5 Flash) provides a free tier sufficient for all Student D sub-problems. Deployment: Render.com, Railway.app, and HuggingFace Spaces all offer free hosting tiers for student projects. Page 42 of 42'},
   0.47048842906951904),
  ({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 38,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem Technique App component A Predict course

In [15]:
question = 'No'
answer_question(question)

Answer:
 I could not find that in the documents.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 24 (similarity 0.153)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 41 (similarity 0.150)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 26 (similarity 0.132)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 35 (similarity 0.119)


('I could not find that in the documents.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 24,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem Technique App component A Train Random Forest, XGBoost, and SVM on 30 URL features. Target: <0.5% false positive rate (blocking legitimate sites is unacceptable). Supervised binary classification URL classification API B Cluster phishing URLs by structural attack pattern (domain spoofing, IP-based, subdomain attack). Profile each attack type. Unsupervised attack pattern mining Attack pattern explorer C Train a character-level CNN/LSTM on raw URL strings without manual features. Show that end-to-end learning is competitive with feature engineering. Deep learning, character-level model Character-level classifier D Given a detected phishing URL, an LLM generates a plain-language explanation of the attack method. Deliver as a Chrome Extension (Manifest V3) call

In [25]:
!pip install -q gradio

In [26]:
import gradio as gr

In [27]:
def process_uploaded_pdf(pdf_file):

    global chunks
    global embeddings
    global index

    if pdf_file is None:
        return "❌ Please upload a PDF first."

    pdf_path = Path(pdf_file)

    # Extract PDF pages
    pages = extract_pdf_pages(pdf_path)

    if not pages:
        return "❌ Could not extract text from this PDF."

    # Create chunks
    new_chunks = []

    for page_data in pages:

        page_chunks = chunk_text(page_data["text"])

        for text_chunk in page_chunks:

            new_chunks.append({
                "source": page_data["source"],
                "page": page_data["page"],
                "text": text_chunk
            })

    if not new_chunks:
        return "❌ No usable text chunks were created."

    # Generate embeddings
    new_embeddings = embedder.encode(
        [item["text"] for item in new_chunks],
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=64
    ).astype("float32")

    # Create new FAISS index
    new_index = faiss.IndexFlatIP(
        new_embeddings.shape[1]
    )

    new_index.add(new_embeddings)

    # Replace existing document data
    chunks = new_chunks
    embeddings = new_embeddings
    index = new_index

    return (
        f"### ✅ Document processed successfully\n\n"
        f"📄 **File:** {pdf_path.name}\n\n"
        f"📑 **Pages:** {len(pages)}\n\n"
        f"🧩 **Chunks:** {len(chunks)}"
    )

In [28]:
def chatbot_response(message, history):

    if not message.strip():
        return "", history

    answer, results = answer_question(message)

    sources = []

    for i, (item, score) in enumerate(results, start=1):

        sources.append(
            f"[{i}] {item['source']} — "
            f"Page {item['page']} "
            f"(similarity: {score:.3f})"
        )

    source_text = "\n".join(sources)

    final_response = (
        f"{answer}\n\n"
        f"**📖 Sources**\n"
        f"{source_text}"
    )

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": final_response
    })

    return "", history

In [29]:
with gr.Blocks(
    title="Document Q&A Assistant"
) as demo:

    # =========================
    # Header
    # =========================

    gr.Markdown("""
    # 📚 Document Q&A Assistant

    Upload a PDF and ask questions using
    **Retrieval-Augmented Generation (RAG)**.
    """)

    # =========================
    # PDF Upload
    # =========================

    gr.Markdown("### 📄 Upload Document")

    with gr.Row():

        pdf_upload = gr.File(
            label="Choose a PDF",
            file_types=[".pdf"],
            type="filepath",
            scale=3
        )

        process_button = gr.Button(
            "⚙️ Process Document",
            variant="primary",
            scale=1
        )

    status = gr.Markdown(
        "Upload a PDF and click **Process Document**."
    )

    # =========================
    # Chat
    # =========================

    gr.Markdown("### 💬 Ask Questions")

    chatbot = gr.Chatbot(
        label="Document Assistant",
        height=500,
        type="messages",
        allow_tags=False
    )

    with gr.Row():

        message = gr.Textbox(
            placeholder="Ask a question about your document...",
            show_label=False,
            lines=2,
            scale=8
        )

        send = gr.Button(
            "➤",
            variant="primary",
            scale=1
        )

    # =========================
    # Examples
    # =========================

    gr.Markdown("### 💡 Try asking")

    gr.Examples(
        examples=[
            "What is the main topic of this document?",
            "What are the key points?",
            "Summarize the document.",
            "What are the main conclusions?",
            "Explain the important concepts."
        ],
        inputs=message
    )

    # =========================
    # Process PDF button
    # =========================

    process_button.click(
        fn=process_uploaded_pdf,
        inputs=pdf_upload,
        outputs=status
    )

    # =========================
    # Send button
    # =========================

    send.click(
        fn=chatbot_response,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    # =========================
    # Enter key
    # =========================

    message.submit(
        fn=chatbot_response,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    # =========================
    # Clear chat
    # =========================

    gr.ClearButton(
        [message, chatbot],
        value="🗑️ Clear Conversation"
    )


# Launch
demo.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_58/3076311804.py:47: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://37397bf95237124d6e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Answer:
 The main topics covered in this document include:

1. **Attention Mechanism**: This section discusses how attention mechanisms are used to "focus" on specific parts of an input while considering others. It explains how attention helps in capturing global context in sequence models.

2. **Transformers**: This part delves into the concept of transformers, which utilize attention mechanisms extensively. It covers various types of attention including self-attention, cross-attention, and positional encoding.

3. **Self-Attention**: This subsection focuses specifically on self-attention, explaining how it allows each token to attend to every other token in the sequence during its computation.

4. **Multi-head Attention (MHA)**: This final section introduces multi-head attention, which involves multiple attention heads to capture different aspects of similarity, enhancing the model's ability to understand complex relationships in data.

Overall, the document provides a comprehensive 